# NB10

Blood-derived module control for tissue specificity.

In [ ]:
# Blood tissue-specificity control

import os, re, gc, glob, time, warnings, traceback
from pathlib import Path
import numpy as np
import pandas as pd
import scipy.stats as stats
import scipy.sparse as sp
from sklearn.decomposition import NMF

import matplotlib
matplotlib.rcParams.update({
    "font.family":"Arial","font.size":8,"axes.titlesize":9,"axes.labelsize":8,
    "xtick.labelsize":7,"ytick.labelsize":7,"legend.fontsize":7,"figure.dpi":150,
    "savefig.dpi":1200,"savefig.bbox":"tight","savefig.pad_inches":0.05,
    "axes.linewidth":0.8,"pdf.fonttype":42,"ps.fonttype":42,
})
import matplotlib.pyplot as plt
try:
    import seaborn as sns; sns.set_style("ticks"); HAS_SNS=True
except ImportError:
    HAS_SNS=False
import scanpy as sc
warnings.filterwarnings("ignore"); np.random.seed(42); SEED=42

BASE_DIR = Path(os.environ.get("MES_BASE_DIR", "."))
PROC_DIR=BASE_DIR/"Process Data"
MANUSCRIPT_DIR=(BASE_DIR/"Manuscript data") if (BASE_DIR/"Manuscript data").exists() else (BASE_DIR/"Manuscript Data")
FIG_DIR=MANUSCRIPT_DIR/"Figures"/"Revision"; FIG_DIR.mkdir(parents=True,exist_ok=True)
TAB_DIR=MANUSCRIPT_DIR/"Tables"/"Revision"; TAB_DIR.mkdir(parents=True,exist_ok=True)
AIM2_DIR=PROC_DIR/"aim2_microglia"; NEG_DIR=PROC_DIR/"negctrl"

run_log=[]
def log(m):
    s=f"[{time.strftime('%H:%M:%S')}] {m}"; print(s,flush=True); run_log.append({"t":time.strftime('%H:%M:%S'),"m":m})
def save_fig(fig,name):
    out=FIG_DIR/f"Supp_Rev_{name}.png"; fig.savefig(out,dpi=1200,bbox_inches="tight"); plt.close(fig); log(f"FIG saved: {out.name}")
def save_xlsx(sheets,name):
    if isinstance(sheets,pd.DataFrame): sheets={"Sheet1":sheets}
    out=TAB_DIR/f"Supp_Rev_{name}.xlsx"
    with pd.ExcelWriter(out,engine="openpyxl") as w:
        for sn,df in sheets.items():
            (df if (df is not None and len(df)) else pd.DataFrame({"note":["no data"]})).to_excel(w,index=False,sheet_name=str(sn)[:31])
    log(f"TAB saved: {out.name}")

TOL_HOMEO=["P2RY12","CX3CR1","TMEM119","GPR34","SALL1","CSF1R","OLFML3"]
TOL_ACTIV=["APOE","SPP1","LPL","TREM2","CST7","CTSD","TYROBP","FCER1G","LGALS3","CD68"]

def looks_log1p(a,n=2000):
    X=a.X; v=(X.data[:min(X.data.size,n)] if X.data.size else np.array([0.])) if sp.issparse(X) else np.asarray(X).ravel()[:n]
    return (np.nanmax(v)<25) and (np.mean(np.abs(v-np.round(v))>1e-6)>0.2)
def umap(vn): return {str(v).upper():str(v) for v in vn}
def present(a,genes):
    m=umap(a.var_names); out=[]
    for g in genes:
        gg=str(g).upper()
        if gg in m and m[gg] not in out: out.append(m[gg])
    return out
def score(a,genes,name,min_g=3):
    pr=present(a,genes)
    if len(pr)<min_g: a.obs[name]=np.nan; return 0
    try: sc.tl.score_genes(a,pr,score_name=name,use_raw=False); return len(pr)
    except Exception: a.obs[name]=np.nan; return 0
def spearman_safe(x,y,minn=20):
    x=pd.to_numeric(pd.Series(x),errors="coerce").to_numpy(); y=pd.to_numeric(pd.Series(y),errors="coerce").to_numpy()
    m=np.isfinite(x)&np.isfinite(y)
    if m.sum()<minn: return np.nan,np.nan,int(m.sum())
    r,p=stats.spearmanr(x[m],y[m]); return float(r),float(p),int(m.sum())
def _X_dense(a,genes):
    X=a[:,genes].X
    return X.tocsr() if sp.issparse(X) else sp.csr_matrix(X)

def to_symbols(a):
    """Convert var_names to gene symbols using common var metadata columns.
    Returns the name of the column used, or None if names already look like symbols."""
    # If var_names already look like symbols (not ENSG/ENSMUSG), keep them.
    sample=[str(v) for v in list(a.var_names[:50])]
    ens_like=sum(1 for v in sample if v.upper().startswith(("ENSG","ENSMUSG","ENS"))) 
    if ens_like < len(sample)*0.5:
        return "already_symbols"
    # find a symbol column
    for col in ["feature_name","gene_symbol","gene_symbols","gene_name","symbol","Symbol",
                "gene_short_name","hgnc_symbol","SYMBOL","features","gene_ids_symbol"]:
        if col in a.var.columns:
            syms=a.var[col].astype(str).values
            # only accept if it yields non-Ensembl-looking names for most genes
            nonens=np.mean([not s.upper().startswith("ENS") and s.lower()!="nan" for s in syms[:200]])
            if nonens>0.5:
                a.var["_orig_id"]=a.var_names
                a.var_names=pd.Index(syms)
                a.var_names_make_unique()
                return col
    return None

# MES gene sets
log("="*72); log("LOAD: MES gene weights")
def _find_gw():
    for d in ["Manuscript data","Manuscript Data","Manuscript_Data","Manuscript","Process Data"]:
        for td in ["Tables","tables","Table",""]:
            for fn in ["Main_Table1.xlsx","Table1.xlsx"]:
                p=(BASE_DIR/d/td/fn) if td else (BASE_DIR/d/fn)
                if p.exists():
                    try:
                        df=pd.read_excel(p,sheet_name="GeneWeights")
                        if any(str(c).startswith("MES") for c in df.columns): return p,df
                    except Exception: pass
    for p in BASE_DIR.rglob("*Table1*.xlsx"):
        try:
            if "GeneWeights" in pd.ExcelFile(p).sheet_names:
                df=pd.read_excel(p,sheet_name="GeneWeights")
                if any(str(c).startswith("MES") for c in df.columns): return p,df
        except Exception: continue
    return None,None
MAIN_T1,df_weights=_find_gw()
if MAIN_T1 is None: raise FileNotFoundError("Main_Table1.xlsx not found")
mes_cols=[c for c in df_weights.columns if str(c).startswith("MES")]
mes_gene_sets={m: df_weights[["gene",m]].dropna().sort_values(m,ascending=False).head(50)["gene"].astype(str).str.upper().tolist() for m in mes_cols}
log(f"  {len(mes_cols)} thymus modules")

# Load microglia cohorts
log("="*72); log("LOAD: scored microglia cohorts")
cohorts={}
for f in sorted(glob.glob(str(AIM2_DIR/"*__microglia_scored.h5ad"))):
    ds=Path(f).name.replace("__microglia_scored.h5ad","")
    try:
        a=sc.read_h5ad(f); a.obs_names_make_unique()
        if "counts" not in a.layers: a.layers["counts"]=a.X.copy()
        if not looks_log1p(a):
            a.X=a.layers["counts"].copy(); sc.pp.normalize_total(a,target_sum=1e4); sc.pp.log1p(a)
        score(a,TOL_HOMEO,"_h"); score(a,TOL_ACTIV,"_a"); a.obs["tolerance_positioning"]=a.obs["_h"]-a.obs["_a"]
        cohorts[ds]=a
        log(f"  {ds}: {a.n_obs:,} cells")
    except Exception as e:
        log(f"  {ds} failed: {e}")

# Derive BLOOD modules in SYMBOL space
log("="*72); log("IDEA 2 (fixed): blood-derived modules in symbol space")
verdict={}
try:
    blood_qc=NEG_DIR/"TS_Blood_NegCtrl_Myeloid__qc.h5ad"
    if not blood_qc.exists():
        hits=list(PROC_DIR.rglob("*Blood*qc.h5ad"))+list(PROC_DIR.rglob("*lood*Myeloid*.h5ad"))
        blood_qc=hits[0] if hits else None
    if blood_qc is None:
        raise FileNotFoundError("TS_Blood QC h5ad not found")
    log(f"  blood data: {blood_qc}")
    blood=sc.read_h5ad(blood_qc); blood.obs_names_make_unique()
    log(f"  blood var_names sample (pre-conversion): {[str(v) for v in list(blood.var_names[:5])]}")
    used=to_symbols(blood)
    log(f"  symbol conversion: {used}; var_names now: {[str(v) for v in list(blood.var_names[:5])]}")
    blood.var_names_make_unique()
    if "counts" not in blood.layers: blood.layers["counts"]=blood.X.copy()
    bt=blood.copy(); bt.X=bt.layers["counts"].copy()
    sc.pp.normalize_total(bt,target_sum=1e4); sc.pp.log1p(bt)
    sc.pp.highly_variable_genes(bt,n_top_genes=2500,flavor="seurat_v3")
    hvg=bt.var_names[bt.var["highly_variable"]].tolist()
    # sanity: how many HVG are symbol-like
    sym_frac=np.mean([not str(g).upper().startswith("ENS") for g in hvg])
    log(f"  blood HVG symbol-like fraction: {sym_frac:.2f} (want ~1.0)")
    rng=np.random.RandomState(SEED)
    idx=rng.choice(bt.n_obs, size=min(bt.n_obs,60000), replace=False) if bt.n_obs>60000 else np.arange(bt.n_obs)
    bt_fit=bt[idx,hvg].copy()
    model=NMF(n_components=8, init="nndsvda", random_state=SEED, max_iter=800)
    model.fit(_X_dense(bt_fit,hvg))
    H=model.components_
    blood_modules={}
    for k in range(8):
        order=np.argsort(H[k])[::-1][:50]
        blood_modules[f"BLOOD{k+1:02d}"]=[str(hvg[i]).upper() for i in order]

    # SANITY GATE: do blood modules map into microglia now?
    test_ds=next(iter(cohorts.values()))
    map_counts=[len(present(test_ds, g)) for g in blood_modules.values()]
    log(f"  blood-module gene mapping into microglia: mean {np.mean(map_counts):.0f}/50 genes")
    if np.mean(map_counts) < 5:
        log("  STILL not mapping (symbol conversion failed). Reporting as untestable, not as null.")
        verdict={"status":"untestable: blood modules still not mapping after conversion",
                 "mean_blood_genes_mapped":float(np.mean(map_counts))}
        save_xlsx({"verdict":pd.DataFrame([verdict])}, "R2b_VERDICT")
        raise SystemExit
    # score + correlate
    rows=[]
    for ds,a in cohorts.items():
        for bk,genes in blood_modules.items():
            score(a,genes,f"_{bk}")
            r,p,n=spearman_safe(a.obs[f"_{bk}"], a.obs["tolerance_positioning"])
            rows.append({"dataset":ds,"module_source":"blood","module":bk,"r":r,"p":p,"n":n})
        for m in mes_cols:
            r,p,n=spearman_safe(a.obs[f"{m}_score"] if f"{m}_score" in a.obs.columns else score(a,mes_gene_sets[m],f"{m}_score") or a.obs.get(f"{m}_score"), a.obs["tolerance_positioning"]) if f"{m}_score" in a.obs.columns else (np.nan,np.nan,0)
            if f"{m}_score" not in a.obs.columns:
                score(a,mes_gene_sets[m],f"{m}_score")
            r,p,n=spearman_safe(a.obs[f"{m}_score"], a.obs["tolerance_positioning"])
            rows.append({"dataset":ds,"module_source":"thymus","module":m,"r":r,"p":p,"n":n})
    df=pd.DataFrame(rows)
    save_xlsx({"thymus_vs_blood_coupling_FIXED":df}, "R2b_TissueSpecificity_Fixed")

    thy=df[df["module_source"]=="thymus"]["r"].abs().dropna()
    bld=df[df["module_source"]=="blood"]["r"].abs().dropna()
    log(f"  N thymus |r| values: {len(thy)}; N blood |r| values: {len(bld)}")
    if len(thy)>=5 and len(bld)>=5:
        U,pmw=stats.mannwhitneyu(thy,bld,alternative="greater")
        # also per-cohort means for transparency
        per=df.groupby(["dataset","module_source"])["r"].apply(lambda s: s.abs().mean()).reset_index()
        log(f"  mean |coupling|: thymus={thy.mean():.3f} (n={len(thy)}), blood={bld.mean():.3f} (n={len(bld)})")
        log(f"  Mann-Whitney (thymus|r| > blood|r|): U={U:.0f}, p={pmw:.4g}")
        specific = bool(pmw<0.05 and thy.mean()>bld.mean())
        verdict={"status":"tested","mean_thymus_abs_r":float(thy.mean()),"mean_blood_abs_r":float(bld.mean()),
                 "MW_U":float(U),"MW_p_thymus_gt_blood":float(pmw),"specificity_supported":specific,
                 "n_thymus":int(len(thy)),"n_blood":int(len(bld))}
        if specific:
            log("  => SPECIFICITY SUPPORTED: thymus modules couple more strongly than blood-derived modules.")
            log("     Genuine strengthening of the central claim; answers Reviewer 3.5 (why thymus).")
        else:
            log("  => NOT specific: blood-derived modules couple comparably to thymus modules.")
            log("     The transcriptional correspondence with microglial tolerance is NOT thymus-specific.")
            log("     This must be disclosed; the 'thymus' framing should be softened to 'myeloid programs'.")
        save_xlsx({"verdict":pd.DataFrame([verdict]),"per_cohort_meanabs_r":per}, "R2b_VERDICT")
        # figure: distribution of |r| thymus vs blood
        fig,ax=plt.subplots(figsize=(4.2,3.4))
        data=[thy.values,bld.values]
        if HAS_SNS:
            import pandas as _pd
            long=_pd.DataFrame({"|coupling r|":list(thy.values)+list(bld.values),
                                "source":["thymus"]*len(thy)+["blood"]*len(bld)})
            sns.boxplot(data=long,x="source",y="|coupling r|",ax=ax,width=0.5,fliersize=2,
                        palette={"thymus":"#D6604D","blood":"#4393C3"})
            sns.stripplot(data=long,x="source",y="|coupling r|",ax=ax,color="black",size=2.5,alpha=0.5)
        else:
            ax.boxplot(data,labels=["thymus","blood"])
        ax.set_title(f"Tissue specificity: thymus vs blood modules\nMW p={pmw:.3g}")
        ax.set_ylabel("|MES-tolerance coupling| (Spearman)")
        for s in ["top","right"]: ax.spines[s].set_visible(False)
        save_fig(fig,"R2b_Specificity_Boxplot")
    else:
        log(f"  insufficient values for MW (thymus {len(thy)}, blood {len(bld)}); reporting untestable.")
        verdict={"status":f"untestable: thymus {len(thy)}, blood {len(bld)} finite |r|"}
        save_xlsx({"verdict":pd.DataFrame([verdict])}, "R2b_VERDICT")
    del blood, bt, bt_fit; gc.collect()
except SystemExit:
    pass
except Exception as e:
    log(f"  IDEA 2 (fixed) FAILED: {e}\n{traceback.format_exc()}")

save_xlsx({"run_log":pd.DataFrame(run_log)}, "MASTER_NB14b_RunLog")
log("="*72); log("NB14b COMPLETE")
